In [ ]:
import pandas as pd
from flask import Flask, request, jsonify
import xgboost as xgb
import pickle



# --- CONFIGURACIÓN FINAL DEL MODELO ---
# El umbral óptimo que maximiza el F1-Score
FINAL_THRESHOLD = 0.3268 
MODEL_FILE = 'Model_XGBOOST.bin'

In [46]:
with open(MODEL_FILE, 'rb') as f_in:
    dv, model = pickle.load(f_in) 

In [71]:
customer = {
    "month": "nov",
    "visitor_type": "returning_visitor",
    "weekend": 1,
    "operating_systems": 2,
    "browser": 2,
    "region": 3,
    "traffic_type":2,
    "administrative":5,
    "administrative_duration":150.0,
    "informational":0,
    "informational_duration":0.0,
    "product_related":30,
    "product_related_duration":2500.0,
    "bounce_rates":0.005,
    "exit_rates": 0.01,
    "page_values":26.0,
    "special_day":0.0
}

for col in ['operating_systems', 'browser', 'region', 'traffic_type']:
    if col in customer:
        customer[col] = str(customer[col])


X = dv.transform([customer])


features = list(dv.get_feature_names_out())

d_X = xgb.DMatrix(X, feature_names=features)
y_pred_prob = model.predict(d_X)[0]


purchase_decision = y_pred_prob >= FINAL_THRESHOLD

result = {
        'purchase_probability': float(y_pred_prob),
        'purchase_decision': bool(purchase_decision),
        'umbral_usado': FINAL_THRESHOLD
    }

result

{'purchase_probability': 0.684770405292511,
 'purchase_decision': True,
 'umbral_usado': 0.3268}